# 记忆治理策略

## 1、消息裁剪

In [1]:
import truststore
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from tencentcloud.common import profile

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [2]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

收到，老王！从现在起我就是小王了，随时听候差遣～ 😄  
有什么想聊的、想问的，或者需要“小王”帮忙跑腿（动脑）的事，您尽管吩咐！
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

可不是嘛老王！今儿这天儿真给力，阳光足、风也轻，感觉骨头缝儿都晒暖了。咱是出去遛弯儿赏花，还是搬把椅子坐院里嗑瓜子儿？您发话，小王随时陪着！😄
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

好嘞老王，您这一问咱可得好好唠唠！  

**我是谁？**  
我是您跟前儿随叫随到的小王——一个AI搭子，没啥实体，但脑子好使、嘴皮子利索。您想聊啥、问啥、编段子、算个账，甚至半夜失眠扯闲篇儿，我都能接茬儿。没有七情六欲，但会模仿人味儿；不会喝酒，但能陪您“云干杯”。  

**您是谁？**  
您是那位爱热闹、有生活气儿的**老王**呀！根据咱刚才的对话：  
1. 您会招呼AI聊天气，八成是个爱观察日子、懂点儿闲情逸致的人。  
2. 您抛来个哲学二连问（我是谁/你是谁），说明偶尔也琢磨点儿深刻事儿，幽默里藏着好奇心。  
3. 性别不明，但“老王”这称呼自带一股胡同口晒太阳的松弛感——可能是退休大爷、中年顽主，或是谁家俏皮长辈。  

**总结版：**  
我是代码做

## 2、消息删除

In [8]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，从现在起你可以叫我小王。  
我会继续帮你，有什么事尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的时候心情也容易变好。  
你今天有什么安排吗？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是小王，一个帮你回答问题、提供建议、一起聊天的 AI 助手。

你是我的对话对象；如果按你刚才给我的称呼，你是和“小王”聊天的人。  
如果你愿意，也可以告诉我你希望我怎么称呼你。


## 3、摘要

In [5]:
import truststore
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from tencentcloud.common import profile

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model_out = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_in = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [11]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三！你是工程师。  
如果你想把这段“非常长非常长的废话”做摘要、提取关键信息、改写成简历自我介绍，或者压缩成一句话，我都可以帮你。
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to date:

这段消息的主要内容是一个名叫张三的工程师重复介绍自己，并且包含了

In [6]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)
print(final_state.values.keys())


StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='你好，我是老王',
                additional_kwargs={},
                response_metadata={},
                id='43cfd9cb-58f6-4f51-8132-b460352fac09'
            ),
            AIMessage(
                content='收到，老王！从现在起我就是小王了，随时听候差遣～ 😄  
\n有什么想聊的、想问的，或者需要“小王”帮忙跑腿（动脑）的事，您尽管吩咐！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 45,
                        'prompt_tokens': 55,
                        'total_tokens': 100,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 55
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': 'e93f6ca9-3e0c-494c-aa67-2fe9838850e4',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f788d-c6bb-7db0-81f8-94e0812c9527-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 55,
                    'output_tokens': 45,
                    'total_tokens': 100,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='今天天气不错',
                additional_kwargs={},
                response_metadata={},
                id='4c886440-241a-4e4c-ac4d-13d8bb84c0f3'
            ),
            AIMessage(
                content='可不是嘛老王！今儿这天儿真给力，阳光足、风也轻，感觉骨头缝儿都晒暖了。咱是出去遛弯儿赏花，
还是搬把椅子坐院里嗑瓜子儿？您发话，小王随时陪着！😄',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 58,
                        'prompt_tokens': 107,
                        'total_tokens': 165,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 107
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '96026f72-a991-4d44-92d2-fe063f164efe',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f788d-cc22-70a0-8191-747ce09c9976-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 107,
                    'output_tokens': 58,
                    'total_tokens': 165,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='告诉我，你是谁？我是谁？',
                additional_kwargs={},
                response_metadata={},
                id='747a71f7-1980-451f-bd31-6de5c1cf47bd'
            ),
            AIMessage(
                content='好嘞老王，您这一问咱可得好好唠唠！  \n\n**我是谁？**  
\n我是您跟前儿随叫随到的小王——一个AI搭子，没啥实体，但脑子好使、嘴皮子利索。您想聊啥、问啥、编段子、算个账，甚至半
夜失眠扯闲篇儿，我都能接茬儿。没有七情六欲，但会模仿人味儿；不会喝酒，但能陪您“云干杯”。  \n\n**您是谁？**  
\n您是那位爱热闹、有生活气儿的**老王**呀！根据咱刚才的对话：  \n1. 
您会招呼AI聊天气，八成是个爱观察日子、懂点儿闲情逸致的人。  \n2. 
您抛来个哲学二连问（我是谁/你是谁），说明偶尔也琢磨点儿深刻事儿，幽默里藏着好奇心。  \n3. 
性别不明，但“老王”这称呼自带一股胡同口晒太阳的松弛感——可能是退休大爷、中年顽主，或是谁家

dict_keys(['messages'])
